# 05 — Classification and Baselines

This notebook trains:
- SVM using a precomputed kernel derived from distance features
- ANN baseline using the same engineered features
- A simple threshold-based baseline

Patient-level train/test splitting is performed before fitting the models.

This notebook does not perform pattern mining.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

PROJECT_ROOT = Path.cwd().parent
FEATURE_DIR = PROJECT_ROOT / "outputs" / "features"
MODEL_DIR = PROJECT_ROOT / "outputs" / "models"
EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

X = np.load(FEATURE_DIR / "distance_features.npy")
y = np.load(FEATURE_DIR / "labels.npy")

print("X:", X.shape)
print("y:", y.shape)


## Patient-level split

The saved feature rows correspond to patient sequences created in Notebook 01.

The split must be performed at patient level. No patient should appear in both training and test data.

For this initial implementation, a stratified 80/20 split is used.


In [ ]:
indices = np.arange(len(y))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())


## SVM with precomputed kernel

The distance representation is converted into a similarity matrix.

A simple RBF transformation over the distance matrix is used to construct a precomputed kernel:

`K(i,j) = exp(-gamma * d(i,j)^2)`

The classifier receives the resulting kernel matrix rather than raw Euclidean features.


In [ ]:
def rbf_distance_kernel(A, B, gamma=1.0):
    # A and B contain distance-to-pattern feature vectors.
    # Their squared Euclidean distance provides a simple custom similarity.
    A2 = np.sum(A * A, axis=1, keepdims=True)
    B2 = np.sum(B * B, axis=1, keepdims=True).T
    squared = np.maximum(A2 + B2 - 2 * A @ B.T, 0.0)
    return np.exp(-gamma * squared)


gamma = 1.0

K_train = rbf_distance_kernel(X_train, X_train, gamma)
K_test = rbf_distance_kernel(X_test, X_train, gamma)

svm = SVC(
    kernel="precomputed",
    class_weight="balanced",
    probability=True,
    random_state=42
)

svm.fit(K_train, y_train)

svm_pred = svm.predict(K_test)
svm_prob = svm.predict_proba(K_test)[:, 1]

print("SVM trained.")


## ANN baseline

The ANN uses the same distance-to-pattern feature vectors so that the comparison focuses on the classifier rather than giving the ANN a different raw representation.


In [ ]:
try:
    import torch
    import torch.nn as nn
except ImportError as exc:
    raise ImportError("Install PyTorch to run the ANN section.") from exc


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Xtr = torch.tensor(X_train_scaled, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
Xte = torch.tensor(X_test_scaled, dtype=torch.float32)

model = nn.Sequential(
    nn.Linear(X_train.shape[1], 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([
        (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
    ], dtype=torch.float32)
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(50):
    model.train()
    optimizer.zero_grad()

    logits = model(Xtr)
    loss = criterion(logits, ytr)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    ann_prob = torch.sigmoid(model(Xte)).numpy().ravel()

ann_pred = (ann_prob >= 0.5).astype(int)

print("ANN trained.")


## Threshold baseline

This is a simple non-sequential baseline using the original clinical variables.

The exact baseline should be kept simple and transparent rather than pretending to reproduce a full clinical scoring system.


In [ ]:
# Placeholder for the threshold baseline.
# Implement this after deciding the exact clinical variables/thresholds
# to use from the available dataset.
threshold_pred = np.zeros_like(y_test)
threshold_prob = threshold_pred.astype(float)

print("Threshold baseline placeholder created.")


In [ ]:
def evaluate_model(name, y_true, pred, prob):
    return {
        "model": name,
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "auroc": roc_auc_score(y_true, prob) if len(np.unique(y_true)) == 2 else np.nan,
        "auprc": average_precision_score(y_true, prob),
    }


results = pd.DataFrame([
    evaluate_model("SVM", y_test, svm_pred, svm_prob),
    evaluate_model("ANN", y_test, ann_pred, ann_prob),
    evaluate_model("Threshold Baseline", y_test, threshold_pred, threshold_prob),
])

results.to_csv(EVAL_DIR / "classification_results.csv", index=False)
display(results)
